# Mixture All — Jordanian scaling (v3)
Run in a Colab GPU runtime. Upload `Idea1-pilot-heads(1).zip` and `zscore_stats.npz` when prompted. The 500 train/200 dev features in Drive are reused; missing Casablanca validation features are now automatically extracted from existing WAVs or the exact pinned Hugging Face validation shards. Extracted features are saved in Drive, so interrupted runs resume without re-encoding cached clips.

**Protocol:** 25 / 50 / 100 unique Jordanian training clips plus 100-example repeat control; three original seeds; frozen AraBEST, fixed supplied z-score and 1,000 head updates. The same 6,819 Casablanca validation rows (one known bad audio row) are scored. No Casablanca data is used for training. Recomputed baseline metrics are compared with the archived metrics, and differences are reported rather than silently substituted.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, getpass, base64

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
REPO = Path('/content/ADI-jordanian-v3')
COMMIT = 'b64437a37e7eda2f1721a56860278640bf085bdd'
URL = 'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git'
if not (REPO / '.git').exists():
    result = subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)],
                            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    if result.returncode:
        token = getpass.getpass('GitHub read-access token (only if repository is private): ')
        auth = base64.b64encode(('x-access-token:' + token).encode()).decode()
        env = {**os.environ, 'GIT_CONFIG_COUNT':'1', 'GIT_CONFIG_KEY_0':'http.extraHeader',
               'GIT_CONFIG_VALUE_0':'Authorization: Basic ' + auth, 'GIT_TERMINAL_PROMPT':'0'}
        subprocess.run(['git','clone','--no-checkout',URL,str(REPO)], env=env, check=True)
        del token, auth, env
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT], check=True)
os.chdir(REPO)
import sys
sys.path.insert(0, str(REPO))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [ ]:
from google.colab import files
import zipfile, io, json, hashlib, time, gc, shutil
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, f1_score
from src.models import acoustic_encoder as ae
from src.experiments import layer_mixture as lm, acoustic_classifier as clf
from src.experiments.casablanca_check import diagnostics

LEVELS = [50, 100]  # optionally add 250
SEEDS = [490, 491, 492]
EXTRACT_BATCH = 1
HEAD_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ROOT / 'output'
RUN = OUT / 'runs/jordanian_scaling_v3' / time.strftime('%Y%m%d-%H%M%S')
RUN.mkdir(parents=True, exist_ok=False)
M = REPO / 'data/manifests'
FULL = ROOT / 'manifests/adi20_micro/c8da454c9f474bb075d3fe608940ee29e8ccfa43/full-8db5842080c2f643/train.jsonl'
AUDIO = ROOT / 'audio/adi20_micro/c8da454c9f474bb075d3fe608940ee29e8ccfa43'

# Find the two supplied files, or ask for them once.
search = [ROOT, Path('/content')]
def locate(pattern):
    return next((p for d in search for p in sorted(d.glob(pattern)) if p.is_file()), None)
if not locate('Idea1-pilot-heads*.zip') or not locate('zscore_stats.npz'):
    os.chdir('/content')
    files.upload()
    os.chdir(REPO)
heads_path, stats_path = locate('Idea1-pilot-heads*.zip'), locate('zscore_stats.npz')
assert heads_path and stats_path, 'Upload both original files.'
heads_zip = zipfile.ZipFile(heads_path)
prefix = next(n[:-len('pilot_run1/config.json')] for n in heads_zip.namelist() if n.endswith('pilot_run1/config.json'))
def original(path):
    return json.loads(heads_zip.read(prefix+path))
CONFIG = original('pilot_run1/config.json')
PILOT = original('pilot_run1/run_record.json')
CAS_RECORD = original('casablanca_check_run3/run_record.json')
LABELS = original('labels.json')['label_order']
JOR = LABELS.index('JOR')
OPT = CONFIG['optimization']
with np.load(stats_path, allow_pickle=False) as s:
    mean, std, layer_names = s['mean'], s['std'], list(s['layer_names'])
assert mean.shape == std.shape == (26,848)
assert np.isfinite(mean).all() and np.isfinite(std).all() and (std>0).all()
assert layer_names == PILOT['features']['layers']
shutil.copy2(stats_path, RUN/'zscore_stats.npz')
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False

# Match the original manifest identities, not merely their sizes.
def read_rows(path, expected=None):
    raw = Path(path).read_bytes()
    assert not expected or hashlib.sha256(raw).hexdigest() == expected, f'Manifest mismatch: {path}'
    return [json.loads(line) for line in raw.splitlines() if line.strip()]
train = read_rows(M/'train_pilot.jsonl', PILOT['data']['train_manifest_sha256'])
dev = read_rows(M/'dev_pilot.jsonl', PILOT['data']['dev_manifest_sha256'])
cas = read_rows(M/'casablanca_validation.jsonl', CAS_RECORD['data']['casablanca_manifest_sha256'])
assert len(train)==500 and Counter(r['label'] for r in train)=={c:25 for c in LABELS}
assert len(dev)==200 and len(cas)==6819
print('Original checkpoints and supplied z-score loaded. Head training device:', HEAD_DEVICE)

In [ ]:
# Original cache: verify identities and read only one feature part at a time.
feature_config = clf.feature_config(CONFIG['encoder'])
prov = lm.expected_provenance(feature_config)
cache = ae.FeatureCache(OUT, prov)
assert (cache.root/'config.json').exists(), f'Original feature cache not found: {cache.root}'
cas_ok = [r for r in cas if r.get('preparation_status','ok')=='ok']
assert len(cas_ok)==6818
for name, rows, expected in [
    ('train',train,PILOT['features']['train_cache_keys_sha256']),
    ('dev',dev,PILOT['features']['dev_cache_keys_sha256'])]:
    missing = cache.missing(rows)
    assert not missing, f'{name}: {len(missing)} original NADI features missing at {cache.root}. This notebook expects the original NADI cache.'
    assert clf.cache_keys_sha256(cache,[r['clip_id'] for r in rows])==expected, f'{name} cache identity mismatch'

print('Casablanca features already cached:', len(cas_ok)-len(cache.missing(cas_ok)), '/', len(cas_ok))

def feature_batches(rows, source, batch_size=64):
    index = source.index()
    groups = defaultdict(list)
    for i,r in enumerate(rows):
        e = index[r['clip_id']]
        assert e['cache_key']==ae.clip_cache_key(r['clip_id'],r['audio_sha256'],source.provenance)
        groups[e['part']].append((i,r['clip_id'],e))
    for part, entries in groups.items():
        with np.load(source.root/part, allow_pickle=False) as data:
            values, ids, keys = data['features'], data['clip_ids'], data['cache_keys']
            for start in range(0,len(entries),batch_size):
                group=entries[start:start+batch_size]
                for _,c,e in group:
                    j=e['row']; assert ids[j]==c and keys[j]==e['cache_key']
                x=values[[e['row'] for _,_,e in group]].astype(np.float32)
                assert np.isfinite(x).all()
                yield [i for i,_,_ in group], x

def load_features(rows, source):
    x = np.empty((len(rows),26,848),np.float32)
    for ids,b in feature_batches(rows,source): x[ids]=b
    return x

def normalize(x):
    x -= mean
    x /= std
    return torch.from_numpy(x)

# Reproduce Fatima’s saved development predictions before training anything.
raw_train = load_features(train,cache)
assert np.allclose(raw_train.mean(0),mean,rtol=1e-4,atol=1e-5), 'Supplied mean does not match the pilot cache'
assert np.allclose(np.maximum(raw_train.std(0),CONFIG['preprocessing']['std_floor']),std,rtol=1e-4,atol=1e-5), 'Supplied std does not match the pilot cache'
x0 = normalize(raw_train)
xd = normalize(load_features(dev,cache))
y0 = torch.tensor([LABELS.index(r['label']) for r in train])
original_heads = {}
for seed in SEEDS:
    path = f'pilot_run1/arms/mixture_all/seed_{seed}/'
    checkpoint = torch.load(io.BytesIO(heads_zip.read(prefix+path+'head.pt')),map_location='cpu',weights_only=True)
    assert checkpoint['layers']==layer_names and checkpoint['labels']==LABELS and checkpoint['standardized']
    assert checkpoint['epochs']==500 and checkpoint['config']==CONFIG
    model = lm.build_model((500,26,848),len(LABELS),0)
    model.load_state_dict(checkpoint['state_dict'],strict=True)
    model.eval()
    ref = {r['clip_id']:r for r in map(json.loads,heads_zip.read(prefix+path+'predictions.jsonl').splitlines())}
    expected = np.array([ref[r['clip_id']]['scores'] for r in dev])
    with torch.no_grad(): actual = model(xd).numpy()
    error = float(np.max(np.abs(actual-expected)))
    assert np.array_equal(actual.argmax(1),expected.argmax(1)) and error<1e-3, f'Baseline reproduction failed for seed {seed}: {error}'
    original_heads[seed] = model
    print('Baseline reproduced:',seed,'maximum logit difference',error)
print('Original normalization verified against the pilot training features.')

In [ ]:
# Use the existing frozen nested 2,000-clip NADI training manifest first.
pool = read_rows(M/'train_scaling_2000.jsonl', '41514213e14bfbd82d4fbf8cfe6eae04a2faf4f878509a15758f431dc41f7929')
if max(LEVELS)>100:
    if not FULL.exists():
        raise FileNotFoundError(f'For >100 Jordanian clips, set FULL to your full NADI train manifest: {FULL}')
    pool += read_rows(FULL, '56a75aa929e2a2c4592012954386cb3d91ff7f256a7004ebe7c4a776573ea4ba')
fields = ('clip_id','audio_sha256','pcm_sha256')
blocked = {k:{r.get(k) for r in train+dev+cas if r.get(k)} for k in fields}
# Exclude any known recording/speaker overlap with evaluation, where metadata exists.
for k in ('source_id','speaker_id'):
    blocked[k] = {r.get(k) for r in dev+cas if r.get(k) and r.get(k)!='unknown'}
extra=[]
rng=np.random.default_rng(490)
# Preserve the frozen scaling pool first, then extend from full train if needed.
for group in (pool[:2000], pool[2000:]):
    candidates=[r for r in group if r.get('label')=='JOR' and r.get('split')=='train']
    for i in rng.permutation(len(candidates)):
        r=candidates[i]
        if any(r.get(k) and r.get(k) in blocked[k] for k in blocked): continue
        extra.append(r)
        for k in fields:
            if r.get(k): blocked[k].add(r[k])
        if len(extra)==max(LEVELS)-25: break
    if len(extra)==max(LEVELS)-25: break
assert len(extra)==max(LEVELS)-25, f'Only {len(extra)} valid additional JOR clips available'
(RUN/'extra_train.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in extra))
print('New unique Jordanian clips:',len(extra))

In [ ]:
# Cache only new Jordanian clips, not the full dataset.
extra_cache = ae.FeatureCache(OUT/'jordanian_extra_cache',prov)
missing = cache.missing(extra)
todo = extra_cache.missing(missing)
if todo:
    assert torch.cuda.is_available(), 'Use a GPU runtime to extract missing Jordanian features.'
    assert AUDIO.is_dir(), f'NADI audio directory missing: {AUDIO}'
    import importlib.util
    if importlib.util.find_spec('speechbrain') is None:
        subprocess.run([sys.executable,'-m','pip','-q','install','speechbrain==1.1.2','huggingface_hub'],check=True)
    encoder = ae.AcousticEncoder.from_pretrained(feature_config,device='cuda',cache_dir='/content/hf-cache')
    clf.frozen_summary(encoder)
    assert json.loads(json.dumps(encoder.provenance())) == json.loads(json.dumps(prov)), 'Encoder provenance mismatch'
    _, failures = clf.prepare_features(encoder,extra_cache,todo,AUDIO,
                         {'batch_size':EXTRACT_BATCH,'chunk_size':8},'extra_JOR')
    del encoder
    gc.collect(); torch.cuda.empty_cache()
    (RUN/'extraction_failures.json').write_text(json.dumps(failures,indent=2))
    assert not failures, 'Some Jordanian extractions failed. See extraction_failures.json.'
xe = np.empty((len(extra),26,848),np.float32)
missing_ids = {r['clip_id'] for r in missing}
for source, indices in [(cache,[i for i,r in enumerate(extra) if r['clip_id'] not in missing_ids]),
                        (extra_cache,[i for i,r in enumerate(extra) if r['clip_id'] in missing_ids])]:
    if indices: xe[indices] = load_features([extra[i] for i in indices],source)
xe = normalize(xe)
print('Ready:',len(x0),'original clips and',len(xe),'new Jordanian clips.')

In [ ]:
# Same model and optimizer, fixed number of updates across experimental arms.
UPDATES = 500 * ((len(train)+OPT['batch_size']-1)//OPT['batch_size'])

def fit(x,y,seed):
    init, shuffle = clf.run_seeds(seed)
    model = lm.build_model(tuple(x.shape),len(LABELS),init).to(HEAD_DEVICE)
    optimizer = lm.make_optimizer(model,OPT)
    x,y = x.to(HEAD_DEVICE),y.to(HEAD_DEVICE)
    gen = torch.Generator().manual_seed(shuffle)
    steps=0
    while steps<UPDATES:
        for ids in torch.randperm(len(x),generator=gen).split(OPT['batch_size']):
            ids=ids.to(HEAD_DEVICE)
            loss=F.cross_entropy(model(x[ids]),y[ids])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),OPT['gradient_clip_norm'])
            optimizer.step(); steps+=1
            if steps==UPDATES: break
    return model.cpu().eval()

models = {('JOR_25_original',s):m for s,m in original_heads.items()}
counts = {'JOR_25_original':(500,25)}
j0 = torch.where(y0==JOR)[0]
for name,k,repeated in [(f'JOR_{k}',k,False) for k in LEVELS]+([('JOR_100_repeat_control',100,True)] if 100 in LEVELS else []):
    added = x0[j0.repeat(3)] if repeated else xe[:k-25]
    x = torch.cat([x0,added]); y = torch.cat([y0,torch.full((k-25,),JOR,dtype=torch.long)])
    counts[name]=(len(y),k)
    for seed in SEEDS:
        model=fit(x,y,seed)
        models[name,seed]=model
        folder=RUN/name/str(seed); folder.mkdir(parents=True,exist_ok=True)
        torch.save({'state_dict':model.state_dict(),'labels':LABELS,'layers':layer_names,
                    'standardized':True,'seed':seed,'updates':UPDATES,'arm':'mixture_all','config':CONFIG},folder/'head.pt')
        print('Trained',name,'seed',seed)
    del x,y,added
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Restore missing Casablanca features (skip this work automatically if cached).
from tempfile import TemporaryDirectory
from pathlib import PurePosixPath
import importlib.util

CAS_REV = CAS_RECORD['data']['casablanca_dataset_version']
CAS_DATASET = 'UBC-NLP/Casablanca'
CAS_AUDIO = ROOT / 'audio/casablanca' / CAS_REV
# Add another directory here if you already have the original dev/<country>/*.wav files.
AUDIO_CANDIDATES = [CAS_AUDIO, Path('/content/casablanca-audio'), ROOT/'casablanca-audio']
CAS_CHUNK = 16
missing_cas = cache.missing(cas_ok)
print(f'Casablanca: {len(cas_ok)-len(missing_cas)}/{len(cas_ok)} cached; {len(missing_cas)} to extract.')

if missing_cas:
    assert torch.cuda.is_available(), 'Select a GPU runtime for Casablanca feature extraction.'
    if importlib.util.find_spec('speechbrain') is None:
        subprocess.run([sys.executable,'-m','pip','-q','install','speechbrain==1.1.2','huggingface_hub'],check=True)
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq
    encoder = ae.AcousticEncoder.from_pretrained(feature_config,device='cuda',cache_dir='/content/hf-cache')
    clf.frozen_summary(encoder)
    assert json.loads(json.dumps(encoder.provenance())) == json.loads(json.dumps(prov)), 'Encoder provenance mismatch'
    failure_log = RUN/'casablanca_extraction_failures.jsonl'

    def extract_rows(rows, audio_root):
        if not rows: return
        _, failures = clf.prepare_features(encoder, cache, rows, audio_root,
                              {'batch_size':EXTRACT_BATCH,'chunk_size':CAS_CHUNK}, 'Casablanca')
        if failures:
            with failure_log.open('a') as f:
                for failure in failures: f.write(json.dumps(failure)+'\n')
            raise RuntimeError(f'{len(failures)} Casablanca extraction failures. See {failure_log}; successfully cached clips will be reused.')

    # Existing prepared audio is preferred: no download required for these rows.
    pending = []
    local_groups = defaultdict(list)
    for r in missing_cas:
        found = next((d for d in AUDIO_CANDIDATES if (d/r['audio_path']).is_file()), None)
        if found is None: pending.append(r)
        else: local_groups[found].append(r)
    for audio_root, rows in local_groups.items():
        extract_rows(rows, audio_root)

    # Download only the validation parquet shards containing uncached clips.
    # Each shard is processed independently, with only a small audio batch in memory.
    by_shard = defaultdict(list)
    for r in pending: by_shard[r['source_shard']].append(r)
    for shard_no, (shard, rows) in enumerate(sorted(by_shard.items()), 1):
        assert 'validation' in shard.lower() and 'test' not in shard.lower(), f'Unexpected shard: {shard}'
        print(f'Shard {shard_no}/{len(by_shard)}: {shard}, {len(rows)} missing clips', flush=True)
        raw_candidates = [ROOT/'raw/casablanca'/CAS_REV/shard, Path('/content/casablanca-raw')/shard]
        existing = next((p for p in raw_candidates if p.is_file()), None)
        with TemporaryDirectory(prefix='casablanca-',dir='/content') as tmp:
            tmp = Path(tmp)
            if existing is None:
                parquet_path = hf_hub_download(repo_id=CAS_DATASET, repo_type='dataset', revision=CAS_REV,
                                               filename=shard, local_dir=str(tmp/'raw'), token=False)
            else:
                parquet_path = existing
            wanted = {int(r['source_row']):r for r in rows}
            assert len(wanted)==len(rows), 'Duplicate source rows in frozen manifest'
            audio_root = tmp/'audio'
            offset=0
            seen=set()
            with pq.ParquetFile(parquet_path) as parquet:
                for batch in parquet.iter_batches(batch_size=CAS_CHUNK, columns=['audio']):
                    selected=[]
                    for i, item in enumerate(batch.to_pylist(),offset):
                        r=wanted.get(i)
                        if r is None: continue
                        payload=item['audio']['bytes']
                        assert payload and hashlib.sha256(payload).hexdigest()==r['audio_sha256'], f'Audio hash mismatch: {r["clip_id"]}'
                        path=audio_root/r['audio_path']
                        path.parent.mkdir(parents=True,exist_ok=True)
                        path.write_bytes(payload)
                        selected.append(r); seen.add(i)
                    offset+=batch.num_rows
                    if selected:
                        extract_rows(selected,audio_root)
                        for r in selected: (audio_root/r['audio_path']).unlink()
                    if len(seen)==len(wanted): break
            assert seen==set(wanted), f'Missing rows from {shard}'
        print('Casablanca cached:',len(cas_ok)-len(cache.missing(cas_ok)),'/',len(cas_ok),flush=True)
    del encoder
    gc.collect(); torch.cuda.empty_cache()

remaining=cache.missing(cas_ok)
assert not remaining, f'{len(remaining)} Casablanca features still missing.'
assert clf.cache_keys_sha256(cache,[r['clip_id'] for r in cas_ok])==CAS_RECORD['features']['casablanca_cache_keys_sha256'], 'Casablanca cache identity mismatch'
print('All Casablanca features ready. Original 6,819-row evaluation can proceed.')


In [ ]:
# Stream Casablanca once across all heads; no 6,819 x 26 x 848 tensor in RAM.
@torch.no_grad()
def predict(model,x):
    return torch.cat([model(b).argmax(1) for b in x.split(64)]).numpy()

cas_predictions={key:np.full(len(cas_ok),-1,dtype=int) for key in models}
with torch.no_grad():
    for ids,b in feature_batches(cas_ok,cache):
        x=normalize(b)
        for key,model in models.items(): cas_predictions[key][ids]=model(x).argmax(1).numpy()
assert all((p>=0).all() for p in cas_predictions.values())
cas_valid=np.array([r.get('preparation_status','ok')=='ok' for r in cas])

def score(rows,pred):
    y=np.array([LABELS.index(r['label']) for r in rows]); pred=np.asarray(pred)
    mask=y==JOR; pos=pred==JOR; present=np.unique(y)
    d=diagnostics(rows,[LABELS[p] if p>=0 else None for p in pred],[LABELS[p] for p in present],LABELS)[0]
    return {'accuracy':float((y==pred).mean()),'macro_f1':float(f1_score(y,pred,labels=present,average='macro',zero_division=0)),
            'jor_recall':float((pred[mask]==JOR).mean()),'jor_precision':float((y[pos]==JOR).mean()) if pos.any() else 0.,
            'jor_predicted_share':float(pos.mean()),'non_jor_accuracy':float((y[~mask]==pred[~mask]).mean()),
            'region_accuracy':d['region_accuracy'],'n':len(y),'failures':int((pred<0).sum())},y

results=[]
baseline_checks=[]
for (arm,seed),model in models.items():
    folder=RUN/arm/str(seed); folder.mkdir(parents=True,exist_ok=True)
    cp=np.full(len(cas),-1,dtype=int); cp[cas_valid]=cas_predictions[arm,seed]
    for split,rows,pred in [('NADI_dev',dev,predict(model,xd)),('Casablanca',cas,cp)]:
        metrics,gold=score(rows,pred)
        if arm=='JOR_25_original' and split=='Casablanca':
            ref=original(f'casablanca_check_run3/arms/mixture_all/seed_{seed}/evaluation/metrics.json')
            check={'seed':seed,'archived_accuracy':ref['accuracy'],'current_accuracy':metrics['accuracy'],
                   'archived_macro_f1':ref['macro_f1'],'current_macro_f1':metrics['macro_f1'],
                   'archived_jor_recall':ref['per_dialect']['JOR']['recall'],'current_jor_recall':metrics['jor_recall']}
            baseline_checks.append(check)
            pd.DataFrame(baseline_checks).to_csv(RUN/'baseline_reproduction.csv',index=False)
            print('Baseline comparison:', check)
            if abs(metrics['accuracy']-ref['accuracy'])>0.005 or abs(metrics['macro_f1']-ref['macro_f1'])>0.005:
                print('WARNING: baseline differs by more than 0.5 percentage points; compare arms against the recomputed baseline and investigate before claiming reproduction.')
        results.append({'arm':arm,'seed':seed,'split':split,'train_size':counts[arm][0],'jor_train_count':counts[arm][1],**metrics})
        names=LABELS+['FAILURE']
        cm=confusion_matrix(gold,np.where(pred<0,len(LABELS),pred),labels=range(len(names)))
        pd.DataFrame(cm,index=names,columns=names).to_csv(folder/f'{split}_confusion.csv')
        pd.DataFrame({'clip_id':[r['clip_id'] for r in rows],'gold':[LABELS[i] for i in gold],
                      'prediction':[LABELS[i] if i>=0 else 'FAILURE' for i in pred]}).to_csv(folder/f'{split}_predictions.csv',index=False)
    np.save(folder/'layer_weights.npy',model.layer_logits.detach().softmax(0).numpy())
    print(arm,seed,'Casablanca Jordanian recall:',round(100*results[-1]['jor_recall'],2),'%')
    pd.DataFrame(results).to_csv(RUN/'results.csv',index=False)
from IPython.display import display
res=pd.DataFrame(results)
metrics=['accuracy','macro_f1','jor_recall','jor_precision','jor_predicted_share','non_jor_accuracy','region_accuracy']
summary=res.groupby(['arm','split'])[metrics].agg(['mean','std'])
summary.to_csv(RUN/'summary.csv')
display((100*summary).round(2))
(RUN/'protocol.json').write_text(json.dumps({'commit':COMMIT,'levels':LEVELS,'seeds':SEEDS,'updates':UPDATES,'head_device':HEAD_DEVICE,
    'baseline':'original saved heads; reproduced dev logits; Casablanca archived versus regenerated metrics in baseline_reproduction.csv','normalization':'supplied original pilot training mean/std, fixed across arms',
    'stats_sha256':hashlib.sha256(stats_path.read_bytes()).hexdigest(),'control':'repeat original JOR 25 x4',
    'test_role':'exploratory Casablanca validation; not blind test','feature_cache':str(cache.root),
    'speaker_independence_verified':False},indent=2))
archive=shutil.make_archive('/content/jordanian_scaling_results_v3','zip',RUN)
print('Results:',archive)
files.download(archive)